# 00 - Setup: Carga de Dados no MongoDB

Este notebook carrega os arquivos CSV da pasta `data/` para o banco de dados **VendasArCondicionado** no MongoDB.

**Pré-requisitos:**
- Docker Compose rodando (`docker compose up -d`)
- MongoDB acessível na porta `27017`
- Dependência `pymongo` instalada no ambiente virtual (certifique-se de rodar `uv sync` para captuar a dependencia)

## 1. Configuração e Conexão

In [1]:
import os
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv(override=True)

DB_USER     = os.getenv('DB_USER', 'admin')
DB_PASSWORD = os.getenv('DB_PASSWORD', '1234')
DB_HOST     = os.getenv('DB_HOST', 'localhost')
DB_PORT     = os.getenv('DB_PORT', '27017')
DB_NAME     = "VendasArCondicionado"

uri = f"mongodb://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/"

try:
    client = MongoClient(uri)
    db = client[DB_NAME]
    client.server_info()
    print(f"Conectado ao MongoDB em {DB_HOST}:{DB_PORT} com sucesso!")
    print(f"Database: {DB_NAME}")
except Exception as e:
    print(f"Erro ao conectar: {e}")

Conectado ao MongoDB em localhost:27017 com sucesso!
Database: VendasArCondicionado


## 2. Criando o Dataset

In [2]:
# 1. Ar Condicionado (Baseado no seu esquema Delta)
ar_condicionados = [
    {"id": 1, "nm_modelo": "Split 12k", "nm_marca": "LG", "valor": 2500.00, "ano": 2024, "potencia": "12000 BTU", "fl_inverter": True},
    {"id": 2, "nm_modelo": "Janela 9k", "nm_marca": "Consul", "valor": 1800.00, "ano": 2023, "potencia": "9000 BTU", "fl_inverter": False},
    {"id": 3, "nm_modelo": "WindFree", "nm_marca": "Samsung", "valor": 3200.00, "ano": 2024, "potencia": "18000 BTU", "fl_inverter": True}
]

# 2. Clientes
clientes = [
    {"cd_cliente": 101, "nome": "Miguel Silva", "cidade": "São Paulo"},
    {"cd_cliente": 102, "nome": "Ana Oliveira", "cidade": "Rio de Janeiro"}
]

# 3. Vendedores
vendedores = [
    {"cd_vendedor": 1, "nome": "João Vendas", "unidade": "Centro"},
    {"cd_vendedor": 2, "nome": "Maria Ar", "unidade": "Norte"}
]

# 4. Vendas (Relacionando os IDs)
vendas = [
    {"id_venda": 1001, "cd_cliente": 101, "cd_vendedor": 1, "id_produto": 1, "data": "2026-04-20", "quantidade": 1},
    {"id_venda": 1002, "cd_cliente": 102, "cd_vendedor": 2, "id_produto": 3, "data": "2026-04-21", "quantidade": 2}
]

print("Datasets preparados para carga.")

Datasets preparados para carga.


## 3. Carga de Dados

In [3]:
dataset = {
    "ar_condicionado": ar_condicionados,
    "clientes": clientes,
    "vendedores": vendedores,
    "vendas": vendas
}

for colecao_nome, dados in dataset.items():
    colecao = db[colecao_nome]
    
    if colecao.count_documents({}) > 0:
        print(f"  ⏭️  {colecao_nome}: já contém registros, pulando...")
    else:
        colecao.insert_many(dados)
        print(f"  ✅ {colecao_nome}: {len(dados)} documentos inseridos.")

print("\n🎉 Carga de dados concluída!")

  ✅ ar_condicionado: 3 documentos inseridos.
  ✅ clientes: 2 documentos inseridos.
  ✅ vendedores: 2 documentos inseridos.
  ✅ vendas: 2 documentos inseridos.

🎉 Carga de dados concluída!


## 4. Validação e Consultas

In [4]:
print(f"{'Coleção':<20} {'Documentos':>10}")
print('-' * 32)

for colecao_nome in dataset.keys():
    count = db[colecao_nome].count_documents({})
    print(f"{colecao_nome:<20} {count:>10}")

print("\n--- AMOSTRA: AR CONDICIONADOS INVERTER ---")
cursor = db.ar_condicionado.find({"fl_inverter": True}, {"_id": 0})
df_sample = pd.DataFrame(list(cursor))
print(df_sample)

Coleção              Documentos
--------------------------------
ar_condicionado               3
clientes                      2
vendedores                    2
vendas                        2

--- AMOSTRA: AR CONDICIONADOS INVERTER ---
   id  nm_modelo nm_marca   valor   ano   potencia  fl_inverter
0   1  Split 12k       LG  2500.0  2024  12000 BTU         True
1   3   WindFree  Samsung  3200.0  2024  18000 BTU         True


In [5]:
client.close()
print('\nConexão encerrada.')


Conexão encerrada.
